In [7]:
import scipy
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import cifar10
from PIL import Image

In [8]:
class DataLoader():
  def __init__(self,dataset_name,img_res=(128,128)):
    self.dataset_name=dataset_name
    self.img_res=img_res

  def load_data(self,batch_size=1,is_testing=False):
    (x_train,_),(x_test,_)=cifar10.load_data()
    x=x_test if is_testing else x_train
    data_type="train" if not is_testing else "test"
    batch_images=np.random.choice(range(x.shape[0]),size=batch_size)
    imgs_hr=[]
    imgs_lr=[]
    for index in batch_images:
      img=x[index,:,:,:]
      h,w=self.img_res
      low_h,low_w=int(h/4),int(w/4)
      img_hr=np.array(Image.fromarray(img).resize((w,h)))
      img_lr=np.array(Image.fromarray(img).resize((low_w,low_h)))
      if not is_testing and np.random.random()<0.5:
        img_hr=np.fliplr(img_hr)
        img_lr=np.fliplr(img_lr)
      imgs_hr.append(img_hr)
      imgs_lr.append(img_lr)
    imgs_hr=np.array(imgs_hr,dtype=np.float32)/127.5-1.
    imgs_lr=np.array(imgs_lr,dtype=np.float32)/127.5-1.
    return (imgs_hr,imgs_lr)

In [9]:
from __future__ import print_function,division
import scipy
import datetime
from keras.layers import BatchNormalization,Activation,Dense,Input,Reshape,Flatten,Dropout,Concatenate,Add,ZeroPadding2D,UpSampling2D,Conv2D
from keras.layers import PReLU,LeakyReLU
from keras.applications import VGG19
from keras.models import Sequential,Model
from keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import keras.backend as k

In [10]:
class SRGAN():
  def __init__(self):
    self.channels=3
    self.lr_height=64
    self.lr_width=64
    self.lr_shape=(self.lr_height,self.lr_width,self.channels)
    self.hr_height=self.lr_height*4
    self.hr_width=self.lr_width*4
    self.hr_shape=(self.hr_height,self.hr_width,self.channels)
    self.n_residential_blocks=16
    self.gf=64
    self.df=64
    self.dataset_name="cifar_dataset"

    opt=Adam(learning_rate=0.0002,beta_1=0.5)

    self.vgg=self.build_vgg()
    self.vgg.trainable=False

    self.data_loader=DataLoader(dataset_name=self.dataset_name,img_res=(self.hr_height,self.hr_width))

    self.discrimator=self.build_discrimator()
    self.discrimator.compile(loss="binary_crossentropy",optimizer=opt,metrics=["accuracy"])

    self.generator=self.build_generator()

    img_lr=Input(shape=self.lr_shape)

    fake_hr=self.generator(img_lr)
    fake_features=self.vgg(fake_hr)

    self.discrimator.trainable=False

    validity=self.discrimator(fake_hr)

    self.combined=Model(img_lr,[validity,fake_features])
    self.combined.compile(loss=["binary_crossentropy","mse"],loss_weights=[1e-3,1],optimizer=opt)

  def build_vgg(self):
    img=Input(shape=self.hr_shape)
    vgg=VGG19(weights="imagenet",include_top=False,input_tensor=img)
    vgg.trainable=False
    img_features=vgg.get_layer("block3_conv4").output
    return Model(inputs=img,outputs=img_features)

  def build_generator(self):
    def residual_block(layer_input,filters):
      d=Conv2D(filters,kernel_size=3,strides=1,padding="same")(layer_input)
      d=Activation("relu")(d)
      d=BatchNormalization(momentum=0.8)(d)
      d=Conv2D(filters,kernel_size=3,strides=1,padding="same")(d)
      d=BatchNormalization(momentum=0.8)(d)
      d=Add()([d,layer_input])
      return d

    def deconv2d(layer_input):
      u=UpSampling2D(size=2)(layer_input)
      u=Conv2D(256,kernel_size=3,strides=1,padding="same")(u)
      u=PReLU(shared_axes=[1,2])(u)
      return u

    img_lr=Input(shape=self.lr_shape)

    c1=Conv2D(64,kernel_size=9,strides=1,padding="same")(img_lr)
    c1=PReLU(shared_axes=[1,2])(c1)

    r=c1

    for _ in range(self.n_residential_blocks):
      r=residual_block(r,self.gf)

    c2=Conv2D(64,kernel_size=3,strides=1,padding="same")(r)
    c2=BatchNormalization(momentum=0.8)(c2)
    c2=Add()([c2,c1])

    u1=deconv2d(c2)
    u2=deconv2d(u1)

    gen_hr=Conv2D(self.channels,kernel_size=9,strides=1,padding="same",activation="tanh")(u2)

    return Model(img_lr,gen_hr)

  def build_discrimator(self):
    def d_block(layer_input,filters,strides=1,bn=True):
      d=Conv2D(filters,kernel_size=3,strides=strides,padding="same")(layer_input)
      d=LeakyReLU(negative_slope=0.2)(d)
      if bn:
        d=BatchNormalization(momentum=0.8)(d)
      return d

    img=Input(shape=self.hr_shape)

    d1=d_block(img,self.df,bn=False)
    d2=d_block(d1,self.df,strides=2)
    d3=d_block(d2,self.df*2)
    d4=d_block(d3,self.df*2,strides=2)
    d5=d_block(d4,self.df*4)
    d6=d_block(d5,self.df*4,strides=2)
    d7=d_block(d6,self.df*8)
    d8=d_block(d7,self.df*8,strides=2)

    d9=Conv2D(self.df*16,kernel_size=3,strides=1,padding="same")(d8)
    d9=LeakyReLU(negative_slope=0.2)(d9)

    validity=Conv2D(1,kernel_size=3,strides=1,padding="same",activation="sigmoid")(d9)

    return Model(img,validity)

  def train(self,epochs,batch_size=1,sample_interval=50):
    start_time=datetime.datetime.now()

    os.makedirs("images/%s"%self.dataset_name,exist_ok=True)

    patch=int(self.hr_height/16)

    for epoch in range(epochs):

      imgs_hr,imgs_lr=self.data_loader.load_data(batch_size)

      fake_hr=self.generator.predict(imgs_lr,verbose=0)

      valid=np.ones((batch_size,patch,patch,1),dtype=np.float32)
      fake=np.zeros((batch_size,patch,patch,1),dtype=np.float32)

      d_loss_real=self.discrimator.train_on_batch(imgs_hr,valid)
      d_loss_fake=self.discrimator.train_on_batch(fake_hr,fake)

      d_loss=0.5*np.add(d_loss_real,d_loss_fake)

      image_features=self.vgg.predict(imgs_hr,verbose=0)

      g_loss=self.combined.train_on_batch(imgs_lr,[valid,image_features])

      elapsed_time=datetime.datetime.now()-start_time

      print("%d [D loss: %f, acc.: %.2f%%] [G loss: %f] time: %s"%(epoch,d_loss[0],100*d_loss[1],g_loss[0],elapsed_time))

      if epoch%sample_interval==0:
        self.sample_images(epoch)

  def sample_images(self,epoch):
    os.makedirs("images/%s"%self.dataset_name,exist_ok=True)

    r,c=2,2

    imgs_hr,imgs_lr=self.data_loader.load_data(batch_size=r,is_testing=True)

    fake_hr=self.generator.predict(imgs_lr,verbose=0)

    imgs_lr=0.5*imgs_lr+0.5
    fake_hr=0.5*fake_hr+0.5
    imgs_hr=0.5*imgs_hr+0.5

    titles=["Generated","Original"]

    fig,axs=plt.subplots(r,c)

    for row in range(r):
      for col,image in enumerate([fake_hr,imgs_hr]):
        axs[row,col].imshow(np.clip(image[row],0,1))
        axs[row,col].set_title(titles[col])
        axs[row,col].axis("off")

    fig.savefig("images/%s/%d.png"%(self.dataset_name,epoch))
    plt.close()

    for i in range(r):
      fig=plt.figure()
      plt.imshow(np.clip(imgs_lr[i],0,1))
      plt.axis("off")
      fig.savefig("images/%s/%d_lowres%d.png"%(self.dataset_name,epoch,i))
      plt.close()

  def save_model(self):
    os.makedirs("saved_model",exist_ok=True)

    def save(model,model_name):
      model_path="saved_model/%s.json"%model_name
      weights_path="saved_model/%s.weights.h5"%model_name

      json_string=model.to_json()

      with open(model_path,"w") as f:
        f.write(json_string)

      model.save_weights(weights_path)

    save(self.generator,"generator")
    save(self.discrimator,"discrimator")

    self.generator.save("saved_model/generator.keras")
    self.discrimator.save("saved_model/discrimator.keras")

In [11]:
!mkdir saved_model

In [12]:
if __name__=="__main__":
  gan=SRGAN()
  gan.train(epochs=100,batch_size=1,sample_interval=10)
  gan.save_model()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1746s 10us/step


/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


0 [D loss: 0.693847, acc.: 23.73%] [G loss: 380.529083] time: 0:30:05.480425
1 [D loss: 0.693395, acc.: 42.72%] [G loss: 252.016663] time: 0:30:18.243891
2 [D loss: 0.693462, acc.: 44.02%] [G loss: 202.864426] time: 0:30:21.314034
3 [D loss: 0.693454, acc.: 45.27%] [G loss: 180.768387] time: 0:30:24.067054
4 [D loss: 0.693389, acc.: 46.03%] [G loss: 169.391525] time: 0:30:26.651937
5 [D loss: 0.693331, acc.: 46.92%] [G loss: 157.558014] time: 0:30:29.271942
6 [D loss: 0.693343, acc.: 46.75%] [G loss: 149.743362] time: 0:30:31.872475
7 [D loss: 0.693323, acc.: 47.24%] [G loss: 138.422272] time: 0:30:35.204486
8 [D loss: 0.693312, acc.: 48.14%] [G loss: 133.433624] time: 0:30:37.870933
9 [D loss: 0.693344, acc.: 48.08%] [G loss: 130.639526] time: 0:30:40.404049
10 [D loss: 0.693336, acc.: 48.22%] [G loss: 128.984650] time: 0:30:43.035694
11 [D loss: 0.693313, acc.: 48.40%] [G loss: 124.397148] time: 0:30:49.027408
12 [D loss: 0.693297, acc.: 48.54%] [G loss: 121.224823] time: 0:30:51.698

In [15]:
import os
os.makedirs("superres",exist_ok=True)
generator_json=gan.generator.to_json()
with open("superres/generator.json","w") as json_file:
  json_file.write(generator_json)
gan.generator.save_weights("superres/generator.weights.h5")

In [16]:
import shutil
shutil.make_archive("superres","zip",".","superres")
from google.colab import files
files.download("superres.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>